# LAB-D2-01: Loss Landscapes and Learning-Rate Roulette

**Purpose:** Connect binary cross-entropy and local slope to visible parameter-update trajectories.

**Objectives:** `OBJ-D2-02`, `OBJ-D2-03`  
**Estimated duration:** 40 minutes live; under 10 seconds compute  
**Prerequisites:** `LESSON-D2-01`, `LESSON-D2-02`, `ACT-D2-01`; Day 1 weighted sums and batch-first arrays  
**Environment:** CPU only; NumPy and matplotlib; deterministic local arrays; no network or download

Workflow: **Observe -> Predict -> Modify -> Run -> Inspect -> Explain -> Extend**. Restart and run in order. The first commitment cell intentionally stops execution until you record your predictions.

In [ ]:
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.22})
print(f"Python {platform.python_version()} | NumPy {np.__version__} | matplotlib {matplotlib.__version__}")
print("Runtime target: local/Colab CPU; no network or GPU required.")

## Recap: From a Forward Pass to an Update

Day 1 ended at a prediction. Training adds an error signal and an update:

$$w_{next}=w-\eta\frac{\partial L}{\partial w}$$

The gradient supplies direction and local sensitivity. The learning rate $\eta$ scales that proposed change. In this lab, loss is evidence for optimization; thresholded accuracy remains a separate evaluation summary.

## Local Dataset: Four Equal-Length BCE Cases

All four cases use targets `y = [1, 0, 1, 0]`. Their probability vectors are generated below, so there is no file or cross-notebook state. Classes use the explicit rule `p >= 0.5`.

In [ ]:
targets = np.array([1.0, 0.0, 1.0, 0.0])
bce_cases = {
    "A": np.array([0.90, 0.10, 0.80, 0.20]),
    "B": np.array([0.55, 0.45, 0.60, 0.40]),
    "C": np.array([0.90, 0.10, 0.49, 0.20]),
    "D": np.array([0.90, 0.10, 0.01, 0.20]),
}
assert targets.shape == (4,)
assert all(values.shape == targets.shape for values in bce_cases.values())
assert all(np.all((0.0 <= values) & (values <= 1.0)) for values in bce_cases.values())
print("Local BCE cases ready:", list(bce_cases))

## Predict Before Calculating BCE

Rank cases `A` through `D` from lowest to highest expected BCE. Record the thresholded accuracy you expect for each case. Then identify one comparison that best exposes information discarded by thresholding. Preserve this commitment when the values appear.

In [ ]:
bce_predictions = {
    "ranking_low_to_high": "",
    "accuracy_A": "",
    "accuracy_B": "",
    "accuracy_C": "",
    "accuracy_D": "",
    "most_informative_comparison": "",
    "reason": "",
}
assert all(value.strip() for value in bce_predictions.values()), (
    "Prediction checkpoint: complete every BCE field before calculating loss."
)

## Modify: Implement Clipped Binary Cross-Entropy

Complete `binary_cross_entropy`. Convert inputs to arrays, clip probabilities to `[epsilon, 1 - epsilon]`, and return the mean per-example BCE. The function must remain finite when supplied probabilities include exact `0` or `1`.

In [ ]:
def binary_cross_entropy(y_true, probabilities, epsilon=1e-12):
    # TODO: clip probabilities and return mean binary cross-entropy.
    raise NotImplementedError("TODO: implement clipped binary cross-entropy")

In [ ]:
bce_results = {}
print(f"{'case':<6} {'accuracy':>10} {'BCE':>12}")
for name, probabilities in bce_cases.items():
    predictions = (probabilities >= 0.5).astype(int)
    accuracy = float(np.mean(predictions == targets))
    loss = binary_cross_entropy(targets, probabilities)
    bce_results[name] = {"accuracy": accuracy, "loss": loss}
    print(f"{name:<6} {accuracy:>10.2f} {loss:>12.6f}")
assert all(np.isfinite(result['loss']) and result['loss'] >= 0.0 for result in bce_results.values())

## Break It Safely: Exact Extreme Probabilities

A hand-written BCE without clipping evaluates logarithms at exact endpoints. Before running the next cell, predict whether the naive result will be finite and state why clipping is a numerical safeguard rather than a claim that the extreme prediction was good.

In [ ]:
extreme_prediction = {"naive_finite": "", "clipping_interpretation": ""}
assert all(value.strip() for value in extreme_prediction.values()), (
    "Prediction checkpoint: commit to the extreme-probability behavior first."
)

In [ ]:
extreme_targets = np.array([1.0, 0.0])
extreme_probabilities = np.array([0.0, 1.0])
with np.errstate(divide='ignore', invalid='ignore'):
    naive_terms = -(extreme_targets * np.log(extreme_probabilities) + (1.0 - extreme_targets) * np.log(1.0 - extreme_probabilities))
    naive_loss = float(np.mean(naive_terms))
stable_extreme_loss = binary_cross_entropy(extreme_targets, extreme_probabilities)
print("Naive extreme BCE:", naive_loss)
print("Clipped extreme BCE:", stable_extreme_loss)
assert not np.isfinite(naive_loss)
assert np.isfinite(stable_extreme_loss) and stable_extreme_loss > 0.0

## Observe: A Deterministic One-Parameter Landscape

Use $L(w)=(w-3)^2$, whose minimum is at `w = 3`. Every run starts at `w = -1` and uses 12 updates. This landscape is a teaching instrument: its numeric learning-rate regimes do not transfer unchanged to other objectives.

## Predict the Four Trajectories

For learning rates `0.01`, `0.10`, `0.90`, and `1.10`, classify the expected path as crawl, smooth convergence, oscillatory convergence, or divergence. Sketch the first three sides/positions relative to `w = 3`, and name the observation that distinguishes shrinking oscillation from divergence.

In [ ]:
learning_rates = [0.01, 0.10, 0.90, 1.10]
trajectory_predictions = {
    "0.01": "",
    "0.10": "",
    "0.90": "",
    "1.10": "",
    "first_three_step_sketch": "",
    "oscillation_vs_divergence_check": "",
}
assert all(value.strip() for value in trajectory_predictions.values()), (
    "Prediction checkpoint: classify and sketch every path before running updates."
)

## Modify: Implement Gradient, Update, and Recorder

Complete the three TODOs. A trajectory dictionary must include arrays named `position`, `loss`, and `distance`, each of length `steps + 1` so the starting state remains visible.

In [ ]:
OPTIMUM = 3.0
START = -1.0
STEPS = 12

def quadratic_loss(position):
    return (position - OPTIMUM) ** 2

def quadratic_gradient(position):
    # TODO: return dL/dw for L(w) = (w - 3)^2.
    raise NotImplementedError("TODO: implement the quadratic gradient")

def gradient_step(position, learning_rate):
    # TODO: apply w_next = w - learning_rate * gradient.
    raise NotImplementedError("TODO: implement one gradient update")

def record_trajectory(start, learning_rate, steps):
    # TODO: record starting state and every update in the required dictionary.
    raise NotImplementedError("TODO: record a complete trajectory")

In [ ]:
trajectories = {rate: record_trajectory(START, rate, STEPS) for rate in learning_rates}
print(f"{'eta':>6} {'first w':>10} {'last w':>12} {'last loss':>14} {'last distance':>16}")
for rate, trace in trajectories.items():
    assert set(trace) == {"position", "loss", "distance"}
    assert all(np.asarray(trace[name]).shape == (STEPS + 1,) for name in trace)
    print(f"{rate:>6.2f} {trace['position'][1]:>10.4f} {trace['position'][-1]:>12.4f} {trace['loss'][-1]:>14.6f} {trace['distance'][-1]:>16.6f}")

## Inspect: Position, Loss, and Distance on Aligned Steps

All panels share the same update axis. Position reveals side-crossing, loss shows optimization progress, and distance distinguishes shrinking from expanding oscillation. The loss/distance panels use logarithmic vertical scales so the divergent path does not flatten every other trace.

In [ ]:
colors = {0.01: "#31688e", 0.10: "#35b779", 0.90: "#f6c85f", 1.10: "#d1495b"}
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
step_axis = np.arange(STEPS + 1)
for rate, trace in trajectories.items():
    label = f"eta={rate:.2f}"
    axes[0].plot(step_axis, trace['position'], marker='o', markersize=3, color=colors[rate], label=label)
    axes[1].plot(step_axis, trace['loss'], marker='o', markersize=3, color=colors[rate], label=label)
    axes[2].plot(step_axis, trace['distance'], marker='o', markersize=3, color=colors[rate], label=label)
axes[0].axhline(OPTIMUM, color='black', linestyle='--', linewidth=1, label='minimum w=3')
axes[0].set(ylabel='parameter position w', title='Same landscape, four update multipliers')
axes[1].set(ylabel='loss L(w)', yscale='log')
axes[2].set(xlabel='update step', ylabel='distance |w - 3|', yscale='log')
for ax in axes:
    ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## Diagnose and Explain the Update Mechanism

For each rate, cite position and distance evidence, then explain the movement using the sign and magnitude of `learning_rate * gradient`. Derive the one-step distance multiplier for this specific quadratic. State why the last loss alone is insufficient to reconstruct every path.

In [ ]:
trajectory_interpretation = {
    "0.01_evidence_and_mechanism": "",
    "0.10_evidence_and_mechanism": "",
    "0.90_evidence_and_mechanism": "",
    "1.10_evidence_and_mechanism": "",
    "distance_multiplier_derivation": "",
    "last_loss_limit": "",
}
assert all(value.strip() for value in trajectory_interpretation.values()), (
    "Diagnosis checkpoint: complete every evidence-and-mechanism field."
)

## Challenge: Predict One New Rate Without a Sweep

Choose one rate in `(0, 1.2)` that is not already present. Use your distance-multiplier derivation to predict the first position, whether side-crossing occurs, and whether distance contracts over six steps. You get one run after committing.

In [ ]:
challenge_rate = None  # TODO: choose one new float in (0, 1.2).
challenge_prediction = {"first_position": "", "side_crossing": "", "six_step_distance_behavior": ""}
assert challenge_rate is not None and 0.0 < float(challenge_rate) < 1.2
assert not any(np.isclose(float(challenge_rate), rate) for rate in learning_rates)
assert all(value.strip() for value in challenge_prediction.values())

In [ ]:
challenge_trace = record_trajectory(START, float(challenge_rate), 6)
print("Challenge positions:", np.round(challenge_trace['position'], 4))
print("Challenge distances:", np.round(challenge_trace['distance'], 4))
challenge_interpretation = ""  # TODO: reconcile the evidence with all three predictions.
assert challenge_interpretation.strip()

## Optional Extension: Change Curvature, Keep the Rate

Replace the landscape with $L_c(w)=c(w-3)^2$ for one positive curvature scale `c`. Before running, predict how the same learning rate changes its effective update multiplier. This extension is not required by the core checkpoint.

In [ ]:
optional_curvature = None  # TODO (optional): choose a positive float other than 1.0.
optional_rate = None  # TODO (optional): choose a positive learning rate.
optional_prediction = ""  # TODO (optional): predict the distance behavior from the new multiplier.
if optional_curvature is None or optional_rate is None or not optional_prediction.strip():
    print("Optional extension skipped. Core checkpoint is unaffected.")
else:
    assert optional_curvature > 0.0 and not np.isclose(optional_curvature, 1.0)
    optional_positions = [START]
    for _ in range(8):
        gradient = 2.0 * optional_curvature * (optional_positions[-1] - OPTIMUM)
        optional_positions.append(optional_positions[-1] - optional_rate * gradient)
    optional_positions = np.asarray(optional_positions)
    optional_distances = np.abs(optional_positions - OPTIMUM)
    print("Optional positions:", np.round(optional_positions, 4))
    print("Optional distances:", np.round(optional_distances, 4))
    optional_interpretation = ""  # TODO (optional): compare prediction with evidence.
    assert optional_interpretation.strip()

## Reflect and Checkpoint

Complete the evidence record. Distinguish what accuracy omitted, what clipping changed numerically, and how update sign/magnitude produced each path. Name one reason these learning-rate thresholds are local to this quadratic.

In [ ]:
reflection = {
    "loss_vs_accuracy": "",
    "clipping_effect": "",
    "update_mechanism": "",
    "transfer_limit": "",
}
assert all(value.strip() for value in reflection.values())
assert np.isfinite(stable_extreme_loss)
assert trajectories[0.01]['distance'][-1] < trajectories[0.01]['distance'][0]
assert trajectories[0.10]['loss'][-1] < trajectories[0.10]['loss'][0]
assert trajectories[0.90]['distance'][-1] < trajectories[0.90]['distance'][0]
assert trajectories[1.10]['distance'][-1] > trajectories[1.10]['distance'][0]
assert np.all(np.sign(trajectories[0.90]['position'][:-1] - OPTIMUM) != np.sign(trajectories[0.90]['position'][1:] - OPTIMUM))
print("LAB-D2-01 checkpoint passed: finite BCE, deterministic paths, aligned evidence, and update-mechanism reflection.")

## Takeaways

- BCE responds to probability quality even when thresholded accuracy ties.
- Clipping keeps a hand-written log loss finite at endpoint probabilities; it does not make a poor prediction acceptable.
- The gradient supplies local direction and sensitivity; the learning rate scales the update.
- Oscillation can contract or expand, so side-crossing alone does not diagnose divergence.
- A trajectory is stronger diagnostic evidence than one final scalar.

## Troubleshooting

| Symptom | Likely cause | Recovery |
|---|---|---|
| BCE is `inf` or `nan` | Logarithm received an endpoint probability | Clip before taking either logarithm |
| Every trajectory is smooth | Gradient or update sign is wrong | Print the first gradient and compare `w - eta * gradient` |
| The `0.90` path does not cross the minimum | The quadratic derivative is missing its factor of two | Differentiate the supplied loss again |
| Arrays have only 12 values | Starting state was not recorded | Store the start, then append 12 updates |
| Plot hides three paths | Divergence controls a linear axis | Keep aligned steps and use the supplied log/symlog scales |
| Results differ after rerunning cells | A supplied constant or trajectory was mutated | Restart the kernel and run from the top |

## Continue

Return to the [LAB-D2-01 debrief](../student-guide/day-2-student-guide.md#lab-d2-01---loss-landscapes-and-learning-rate-roulette). Review [LESSON-D2-01](../student-guide/day-2-student-guide.md#lesson-d2-01---the-complete-learning-loop), [LESSON-D2-02](../student-guide/day-2-student-guide.md#lesson-d2-02---loss-gradient-descent-and-learning-rate), and [ACT-D2-01](../challenges/day-2-challenges.md#act-d2-01---loss-ranking-same-accuracy-different-signal) as needed. Environment and notation conventions are in the [shared environment](../../shared/environment.md) and [notation contract](../../shared/notation-and-style.md).